# media-ingest: Pipeline Prototyping

Test chunking and embedding on transcription data.

```bash
pip install -e libs/dagster-io -e packages/media-ingest
```

In [ ]:
import os
# os.environ["LLM_BASE_URL"] = "http://litellm.your-cluster:4000/v1"
# os.environ["LLM_API_KEY"] = "sk-..."
# os.environ["EMBEDDING_PROVIDER"] = "huggingface"  # for local embeddings
# os.environ["EMBEDDING_MODEL"] = "all-MiniLM-L6-v2"

## 1. Simulate Transcription Output

In [ ]:
# Simulated output from the Whisper transcription stage
sample_transcriptions = [
    {
        "document_id": "media-metube-interview-001",
        "title": "Tech Interview Episode 42",
        "text": (
            "Welcome to the show. Today we're talking about large language models and their "
            "applications in data engineering. Our guest is Dr. Sarah Chen from Stanford "
            "University. Sarah, you've been working on retrieval-augmented generation for "
            "structured data. Can you tell us about your recent paper? "
            "Sure, the key insight is that chunking strategy matters enormously for RAG quality. "
            "We found that recursive character splitting with semantic boundaries outperforms "
            "fixed-size chunking by 23% on retrieval benchmarks. The overlap between chunks "
            "is critical for maintaining context across boundaries."
        ),
        "language": "en",
        "segments": 15,
    },
    {
        "document_id": "media-tubesync-lecture-002",
        "title": "Database Systems Lecture 7",
        "text": (
            "Today we're covering vector databases and approximate nearest neighbor search. "
            "The fundamental data structure is the HNSW graph, which provides logarithmic "
            "search complexity. Popular implementations include FAISS from Meta, Pinecone, "
            "and Weaviate. For our purposes, we'll focus on pgvector which integrates "
            "directly with PostgreSQL. The key parameters are the number of lists for IVF "
            "indexing and the ef_search parameter for HNSW."
        ),
        "language": "en",
        "segments": 12,
    },
    {
        "document_id": "media-metube-failed-003",
        "title": "Corrupted Audio File",
        "text": "",
        "language": "unknown",
        "error": "Whisper failed: audio too short",
    },
]

print(f"Transcriptions: {len(sample_transcriptions)}")
for t in sample_transcriptions:
    status = f"{len(t['text'])} chars" if t["text"] else f"FAILED: {t.get('error')}"
    print(f"  {t['document_id']}: {status}")

## 2. Chunking

In [ ]:
from dagster_io import chunk_document

all_chunks = []
skipped = 0

for t in sample_transcriptions:
    if not t.get("text"):
        skipped += 1
        continue
    chunks = chunk_document(
        document_id=t["document_id"],
        title=t.get("title", ""),
        content=t["text"],
        metadata={"source": "media_ingest", "language": t.get("language", "unknown")},
        chunk_size=250,
        chunk_overlap=50,
    )
    all_chunks.extend(chunks)

print(f"{len(sample_transcriptions)} transcriptions → {len(all_chunks)} chunks (skipped {skipped})")
for chunk in all_chunks:
    print(f"\n{chunk.chunk_id} ({len(chunk.text)} chars):")
    print(f"  {chunk.text[:100]}...")

## 3. Embedding & Retrieval Test

In [ ]:
from dagster_io import EmbeddingResource
import numpy as np

emb = EmbeddingResource()
emb.setup_for_execution(None)

texts = [c.text for c in all_chunks]
vectors = emb.embed(texts)

print(f"Embedded {len(vectors)} chunks → {len(vectors[0])}d")

# Semantic search test
queries = [
    "RAG chunking strategies",
    "vector database indexing",
    "large language models",
]

for query in queries:
    qvec = emb.embed_single(query)
    sims = [
        np.dot(qvec, v) / (np.linalg.norm(qvec) * np.linalg.norm(v))
        for v in vectors
    ]
    best_idx = int(np.argmax(sims))
    print(f"\nQuery: '{query}'")
    print(f"  Best match ({sims[best_idx]:.3f}): {all_chunks[best_idx].chunk_id}")
    print(f"  Text: {all_chunks[best_idx].text[:80]}...")

## 4. Embedding Model Comparison

Compare OpenAI vs HuggingFace local embeddings (requires `dagster-io[huggingface]`).

In [ ]:
# Uncomment to compare embedding models
#
# from dagster_io import EmbeddingResource
#
# configs = [
#     {"provider": "openai", "model": "text-embedding-3-small"},
#     {"provider": "huggingface", "model": "all-MiniLM-L6-v2"},
#     {"provider": "huggingface", "model": "all-mpnet-base-v2"},
# ]
#
# for cfg in configs:
#     emb = EmbeddingResource(**cfg)
#     emb.setup_for_execution(None)
#     vecs = emb.embed(texts[:2])
#     sim = np.dot(vecs[0], vecs[1]) / (np.linalg.norm(vecs[0]) * np.linalg.norm(vecs[1]))
#     print(f"{cfg['provider']}/{cfg['model']}: dims={len(vecs[0])}, similarity={sim:.4f}")